#### 基于Qwen2.5-0.5B 基础预训练模型和LoRA微调的的用户交易意图分析分类微调模型

##### 加载Qwen2.5-0.5B模型
 - 使用ModelScope对Transformers包的封装加载Transformers的AutoModel和AutoTokenizer;
 - 分别加载模型和分词器
##### 调整模型输出层
- 添加一个分类层
- 维度为896、in_features=896、out_features=2

In [ ]:
from modelscope import AutoModel, AutoTokenizer

device = "cuda"

model_path = "Qwen/Qwen2.5-0.5B"

model = AutoModel.from_pretrained(pretrained_model_name_or_path=model_path).to(device)
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_path)

2026-03-16 16:39:05,193 - modelscope - INFO - Target directory already exists, skipping creation.
Some weights of Qwen2ForSequenceClassification were not initialized from the model checkpoint at /root/.cache/modelscope/hub/models/Qwen/Qwen2___5-0___5B and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


2026-03-16 16:39:13,013 - modelscope - INFO - Target directory already exists, skipping creation.


##### 模型测试
- 使用cuda0设备、token IDS、res_token_ids转PyTorch Tensor

In [3]:
token_ids = tokenizer(["这个活动的后续流程是怎样的？"], return_tensors="pt", padding=True, truncation=True).to(device)
token_ids.input_ids[-1, :]
res_token_ids = model(**token_ids)
res_token_ids

SequenceClassifierOutputWithPast(loss=None, logits=tensor([[ -2.6723, -12.5859]], device='cuda:0', grad_fn=<IndexBackward0>), past_key_values=DynamicCache(layers=[DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer]), hidden_states=None, attentions=None)

##### 加载数据集
- 数据预处理，需要拼接活动类型、标签节点到用户输入中；
- 构造平衡数据集（平衡数据集分布按照数量少的进行平衡性划分）：先取得更少label的数据集数量，再将其作为采样更多label的数据集的依据，最后将采样过后的数据集和更少的数据集合并即可；
- 创建训练集、验证集和测试集：根据7:1:2的比例构造数据集，注意先重置索引，因为需要根据比例计算两个数据集的结束位置，最后直接从上一个结束位置加载剩余数据集即可；
- 分批加载训练集、验证集、测试集；
- 数据预处理：对数据进行padding或truncation并将数据文本部分分别进行token化；

In [4]:
# 数据预处理

import pandas as pd
from pandas.core.frame import DataFrame

source_dataset_df = pd.read_csv(filepath_or_buffer="./trading_intent_classification_lora/lora_adapter/source_datasets.csv")
datasets = []

for i in range(0, len(source_dataset_df)):
    # 需要对数据集进行拆分（根据type进行拆分）
    split_result = source_dataset_df['type'][i].split("、")
    for type_item in split_result:
        tag = ""
        if type_item == "讲好中国故事":
            tag = source_dataset_df['tag_b'][i]
        elif type_item == "诗词少年":
            tag = source_dataset_df['tag_a'][i]
        
        if tag != "":
            datasets.append([f"Text:{source_dataset_df['text'][i]}\\nType:{type_item}\\nTag:{tag}", source_dataset_df['label'][i]])
pd.DataFrame(data=datasets, columns=['Text', 'Label']).to_csv(path_or_buf="./trading_intent_classification_lora/lora_adapter/datasets.csv", index=None)

In [5]:
# 构造平衡数据集

dataset_df = pd.read_csv(filepath_or_buffer="./trading_intent_classification_lora/lora_adapter/datasets.csv", sep=",", header=None, names=["Text", "Label"])
dataset_df['Label'].value_counts()

def create_balance_df(df: DataFrame) -> DataFrame:
    """构造平衡数据集，先根据数量少的label确定另一种label的采样数量，再对双方采样相同数量数据集即可

    Args:
        df (DataFrame): 读取的CSV原始数据集

    Returns:
        DataFrame: 平衡数据集
    """
    num_has_intent = df[df['Label'] == "0"].shape[0] # 读取数量更少的label类型
    has_no_intent_subset = df[df['Label'] == "1"].sample(n=num_has_intent, random_state=312) # 根据数量更少的label类型随机采样另一种类型的数据集
    
    return pd.concat(objs=[has_no_intent_subset, df[df['Label'] == "1"]]) # 拼接两批数据即可
    
balance_dataste = create_balance_df(dataset_df)
balance_dataste['Label'] = balance_dataste['Label'].map({"1":1, "0":0}) # 数据类型转换
balance_dataste

,Text,Label
78,Text:家里二宝一起报名，能享受亲情价吗？\nType:讲好中国故事\nTag:预选赛晋级(总),1
2041,Text:想了解报名后能不能预约参赛时间？\nType:讲好中国故事\nTag:预选赛晋级(总),1
631,Text:请问报名后包含哪些学习保障？\nType:诗词少年\nTag:(总)复选已晋级,1
2040,Text:想了解报名后能不能预约参赛时间？\nType:诗词少年\nTag:(总)复选已晋级,1
874,Text:城市擂台赛一般需要多久到现场就可以？\nType:讲好中国故事\nTag:预选赛晋...,1
...,...,...
2233,Text:晋级后不去，会不会产生违约金？\nType:讲好中国故事\nTag:预选赛晋级(总),1
2234,Text:请问报名费包含哪些服务内容？\nType:诗词少年\nTag:(总)复选已晋级,1
2235,Text:请问报名费包含哪些服务内容？\nType:讲好中国故事\nTag:预选赛晋级(总),1
2236,Text:报名成功后，还有别的费用吗？\nType:诗词少年\nTag:(总)复选已晋级,1


In [6]:
# 构造训练集、验证集和测试集
def dataset_random_split(df: DataFrame, train_frac: float, validation_frac: float) -> tuple[DataFrame, DataFrame, DataFrame]:
    """按照比例构造训练集、验证集和测试集（测试集比例自动推断）

    Args:
        df (DataFrame): 平衡数据集
        train_frac (float): 训练集比例
        validation_frac (float): 验证集比例

    Returns:
        tuple[DataFrame, DataFrame, DataFrame]: 训练集、验证集和测试集
    """
    df = df.sample(frac=1, random_state=123).reset_index(drop=True) # 重新设置索引
    train_sample_end_idx = int(len(df) * train_frac) # 设置训练集结束索引位置
    val_sample_end_idx = train_sample_end_idx + int(len(df) * validation_frac) # 设置验证集结束索引位置
    
    return df[:train_sample_end_idx], df[train_sample_end_idx: val_sample_end_idx], df[val_sample_end_idx:]

dataset_path = {
    "train": "./trading_intent_classification_lora/lora_adapter/train.csv",
    "val": "./trading_intent_classification_lora/lora_adapter/val.csv",
    "test": "./trading_intent_classification_lora/lora_adapter/test.csv"
}
# 训练集:验证集:测试集=7:1:2
train_dataset_df, val_dataset_df, test_dataset_df = dataset_random_split(df=balance_dataste, train_frac=0.8, validation_frac=0.1)
train_dataset_df.to_csv(path_or_buf=dataset_path['train'], index=None)
val_dataset_df.to_csv(path_or_buf=dataset_path['val'], index=None)
test_dataset_df.to_csv(path_or_buf=dataset_path['test'], index=None)

In [7]:
from datasets import load_dataset

dataset = load_dataset("csv", data_files=dataset_path)

tokenized_datasets = dataset.map(
    function=lambda examples: tokenizer(examples['Text'], truncation=True, max_length=512, padding=True),
    batched=True,
    remove_columns=['Text']
)

tokenized_datasets = tokenized_datasets.rename_column("Label", "labels")

Generating train split: 0 examples [00:00, ? examples/s]

Generating val split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1789 [00:00<?, ? examples/s]

Map:   0%|          | 0/223 [00:00<?, ? examples/s]

Map:   0%|          | 0/225 [00:00<?, ? examples/s]

##### 初始化LoRA微调
预计微调500万参数，会自动冻结除target_modules以外的层
- 注意安装PEFT包（Powered By Transformers）；
- 初始化LoraConfig，包含任务类型（因果语言模型，即Transformers）；
- 设置rank以及缩放、设置暂退率为0.1同时配置需要融合LoRA层的原始模型层（应用到全部线性层）
- 创建LoRA Layer、LoRA With Linear、replace_linear_with_lora

In [8]:
from peft import LoraConfig, get_peft_model, TaskType

model.train()

lora_conf = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=8,     # LoRA Alpha（参数a，微调初始阶段使用默认值）
    lora_dropout=0.08, # LoRA微调暂退率（LoRA微调初始阶段使用默认值）
    # target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "score"],
    target_modules=["q_proj", "k_proj"], # 目标模型（初始阶段选择核心Q/V+顶层2~4层FFN）
    modules_to_save=["score"],
    inference_mode=False # 表示当前是训练模式，若为True则是在推理模式下的一些特定优化会被应用
)

model=get_peft_model(model=model, peft_config=lora_conf) # 会自动冻结除target_modules以外的层
model.print_trainable_parameters()

trainable params: 542,464 || all params: 494,577,024 || trainable%: 0.1097


##### 配置训练参数
配置的训练参数包含以下内容：
- 设置模型训练检查点、结果的保存路径，已经覆盖已存在的输出目录；
- 需要训练过程中执行训练和评估；
- 超参设置，包括迭代次数、学习率、训练和验证集的批次大小、权重衰减系数；
- 设置模型的执行策略，包括每次epoch结束后进行一次验证、每次epoch结束后保存检查点、检查点的默认保存策略为每次epoch结束后；
- 注意：如果需要打印Training Loss，需要配置logging_strategy=epoch/steps；

In [9]:
from transformers import (
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score
import numpy as np
import torch

model.config.pad_token_id = tokenizer.pad_token_id

training_args = TrainingArguments(
    output_dir="./trading_intent_classification_lora",       # 模型训练检查点、成果的保存路径
    overwrite_output_dir=True,                               # 覆盖已存在的输出目录
    # learning_rate=1e-4,                                      # 学习率（初始微调阶段采用默认值）
    per_device_train_batch_size=20,                          # 训练集批次大小
    per_device_eval_batch_size=20,                           # 验证集批次大小
    num_train_epochs=1,                                      # 迭代次数
    # max_steps=20,                                            # 迭代次数（改用steps）
    # weight_decay=0.01,                                     # 权重衰减系数（暂不设置）
    logging_strategy="steps",                                # 每个epoch结束后记录日志（如果配置为steps需要设置logging_steps参数）
    eval_strategy="steps",                                   # 验证过程在每次epoch结束后执行
    save_strategy="steps",                                   # 在每次epoch结束后保存checkpoint
    metric_for_best_model="accuracy",                        # 选择最佳模型的指标（准确率）
    greater_is_better=True,                                  # 评估指标越大越好（True）
    report_to="tensorboard",                                 # 启动TensorBoard
    run_name="trading_intent_classification",                # 运行名称
    logging_dir="./trading_intent_classification_lora/logs",  # 日志目录
    load_best_model_at_end=True,
    logging_steps=5
)

def compute_metrics(eval_pred) -> dict[str, float]:
    """模型评估指标计算

    Args:
        eval_pred (_type_): 模型训练输出

    Returns:
        dict[str, float]: 模型指标字典
    """
    
    logits, labels = eval_pred
    predictions = np.argmax(a=logits, axis=-1)
    
    return {"accuracy": accuracy_score(y_true=labels, y_pred=predictions)}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["val"],
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer, padding=True, return_tensors="pt"),
    compute_metrics=compute_metrics
)

##### 开始训练模型
- 需要设置train的resume_from_checkpoint=True，使训练可自动加载ouput_dir中的最新检查点
- 注意：如果一开始没有检查点，则无法进行训练，需要先开启训练，后续有加载需要时再配置resume_from_checkpoint参数

##### TensorBoard
- 使用TensorBoard进行训练日志、数据可视化，是TensorFlow的可视化工具，与Transformers原生集成；
- 需要安装依赖（pip install tensorboard）；
- 在TrainingArguments中需要增加参数report_to="tensorboard"；

##### 超参调整
- train loss快速下降、val loss高 -> rank过大产生过拟合，减小rank；
- 高rank表示高自由度，能学习到样本级别噪声、prompt偏置、token位置偶然相关性，数据集中的特有模式等；
- 低rank则使模型只学最重要的方向；
- 一次只改变一个超参，先确认最重要的容量参数（rank+target module）是否合理，再微调其他超参；
- LoRA微调的初始阶段，先用默认值设置a、dropout、learning_rate，先验证rank（8~16）初始和target module的合理性（核心Q/V+顶层2~4层FFN），待训练曲线率、dropout或scaling进行微调；
- 训练观察阶段：小规模试验；

In [10]:
trainer.train()

Step,Training Loss,Validation Loss


KeyboardInterrupt: 

##### 模型测试
使用测试集中的数据测试模型的准确率

In [ ]:
tests_df = val_dataset_df.reset_index(drop=True)

all_samples = len(tests_df)
correct_num = 0
model.eval()

for i in range(0, all_samples):
    token_ids = tokenizer([tests_df['Text'][i]], return_tensors="pt", padding=True, truncation=True).to(device)
    if accuracy_score(y_pred=torch.argmax(input=model(**token_ids).logits, dim=-1).to("cpu").numpy(), y_true=[tests_df['Label'][i]]) == 1.0:
        correct_num+=1
accuracy = f"Test Accuracy:{correct_num / all_samples}"
accuracy

'Test Accuracy:1.0'

##### 保存和加载模型
- 使用model.save_pretrained(save_path)保存模型
- 使用tokenizer.save_pretrained(save_path)保存对应的分词器
- 使用PeftModel.from_pretrained(base_model, save_path)加载并使用

In [ ]:
from peft import PeftModel

save_directory = "./trading_intent_classification_lora/lora_adapter"
merged_model = model.merge_and_unload()
merged_model.save_pretrained(save_directory=save_directory)
tokenizer.save_pretrained(save_directory=save_directory)

2026-03-16 16:38:14.933275: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-16 16:38:15.200252: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-16 16:38:18.898480: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


NameError: name 'model' is not defined

In [ ]:
# 模型上传
!modelscope upload "modelscope_mp_260903038/ckcz-trading-intent" "./trading_intent_classification_lora/lora_adapter" --token ms-8c6ec26a-dcf8-4df7-984d-a5f30b5bd026


 _   .-')                _ .-') _     ('-.             .-')                              _ (`-.    ('-.
( '.( OO )_             ( (  OO) )  _(  OO)           ( OO ).                           ( (OO  ) _(  OO)
 ,--.   ,--.).-'),-----. \     .'_ (,------.,--.     (_)---\_)   .-----.  .-'),-----.  _.`     \(,------.
 |   `.'   |( OO'  .-.  ',`'--..._) |  .---'|  |.-') /    _ |   '  .--./ ( OO'  .-.  '(__...--'' |  .---'
 |         |/   |  | |  ||  |  \  ' |  |    |  | OO )\  :` `.   |  |('-. /   |  | |  | |  /  | | |  |
 |  |'.'|  |\_) |  |\|  ||  |   ' |(|  '--. |  |`-' | '..`''.) /_) |OO  )\_) |  |\|  | |  |_.' |(|  '--.
 |  |   |  |  \ |  | |  ||  |   / : |  .--'(|  '---.'.-._)   \ ||  |`-'|   \ |  | |  | |  .___.' |  .--'
 |  |   |  |   `'  '-'  '|  '--'  / |  `---.|      | \       /(_'  '--'\    `'  '-'  ' |  |      |  `---.
 `--'   `--'     `-----' `-------'  `------'`------'  `-----'    `-----'      `-----'  `--'      `------'

2026-03-16 14:00:18,091 - modelscope - WARNING - Repo